In [1]:
# 6_add_vector_embedding.ipynb
#
# Embeds every natural-language profile from step 5_add_nl_strings using the OpenAI
# text-embedding-3-small model (1 536 dimensions).
#
# Input:  data/4_feature_eng/{WAVE}_feature_eng.pkl  (nl_profile column added by 5_add_nl_strings)
# Output: data/6_add_vector_embedding/{WAVE}_vector_embedding.pkl + .csv (this step)
#         data/6_embeddings/ — embeddings.npy, embeddings_index.csv, embeddings.parquet (step 8)
#
# Requires OPENAI_API_KEY in environment or .env file.
# Embeddings are batched (512 texts per call) and checkpointed every
# CHECKPOINT_EVERY batches so long runs can be resumed after interruption.

import sys, os, time, json
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_variables as _cv_df
importlib.reload(_cv_df)
from data_pipeline.config_variables import DATA_FOLDER, WAVE

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
    print("Loaded .env")
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_MODEL      = "text-embedding-3-small"   # 1 536 dims, $0.02 / 1M tokens
EMBED_DIMS       = 1_536
BATCH_SIZE       = 512    # texts per API call (max 2 048)
CHECKPOINT_EVERY = 10     # save partial results every N batches
RETRY_LIMIT      = 3
RETRY_DELAY      = 5      # seconds between retries

INPUT_PKL = Path(f"../{DATA_FOLDER}/4_feature_eng/{WAVE}_feature_eng.pkl")
LEGACY_EMBED_DIR = Path(f"../{DATA_FOLDER}/6_embeddings")
ARTIFACT_DIR     = Path(f"../{DATA_FOLDER}/6_add_vector_embedding")
OUTPUT_NPY       = LEGACY_EMBED_DIR / "embeddings.npy"
OUTPUT_IDX_CSV   = LEGACY_EMBED_DIR / "embeddings_index.csv"
OUTPUT_PARQUET   = LEGACY_EMBED_DIR / "embeddings.parquet"
OUTPUT_PKL       = ARTIFACT_DIR / f"{WAVE}_vector_embedding.pkl"
OUTPUT_CSV       = ARTIFACT_DIR / f"{WAVE}_vector_embedding.csv"
CHECKPOINT_NPY   = LEGACY_EMBED_DIR / "_checkpoint.npy"
CHECKPOINT_IDX   = LEGACY_EMBED_DIR / "_checkpoint_idx.csv"

LEGACY_EMBED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Load profiles ─────────────────────────────────────────────────────────────
print(f"Reading: {INPUT_PKL}")
df = pd.read_pickle(INPUT_PKL)[["pidp", "nl_profile"]].copy()
print(f"Loaded {len(df):,} profiles")

n_before = len(df)
df = df[df["nl_profile"].notna() & (df["nl_profile"].str.strip() != "")].reset_index(drop=True)
if len(df) < n_before:
    print(f"Dropped {n_before - len(df):,} rows with empty profiles")

texts     = df["nl_profile"].tolist()
n_total   = len(texts)
est_cost  = n_total * 80 / 1_000_000 * 0.02
print(f"\nModel: {EMBED_MODEL}  |  Profiles: {n_total:,}  |  Est. cost: ~${est_cost:.3f} USD")

# ── Resume from checkpoint if one exists ─────────────────────────────────────
if CHECKPOINT_NPY.exists() and CHECKPOINT_IDX.exists():
    embeddings_list = list(np.load(CHECKPOINT_NPY))
    start_idx       = len(pd.read_csv(CHECKPOINT_IDX))
    print(f"Resuming from checkpoint: {start_idx:,} already embedded")
else:
    embeddings_list = []
    start_idx       = 0
    print("Starting fresh")


def embed_batch(batch_texts: list[str], attempt: int = 0) -> list[list[float]]:
    try:
        response = client.embeddings.create(input=batch_texts, model=EMBED_MODEL)
        return [item.embedding for item in response.data]
    except Exception as exc:
        if attempt < RETRY_LIMIT:
            print(f"  API error ({exc}); retrying in {RETRY_DELAY}s ...")
            time.sleep(RETRY_DELAY)
            return embed_batch(batch_texts, attempt + 1)
        raise


# ── Embed ─────────────────────────────────────────────────────────────────────
remaining = texts[start_idx:]
with tqdm(total=len(remaining), desc="Embedding", unit="profile") as pbar:
    for batch_num, batch_start in enumerate(range(0, len(remaining), BATCH_SIZE)):
        batch = remaining[batch_start : batch_start + BATCH_SIZE]
        embeddings_list.extend(embed_batch(batch))
        pbar.update(len(batch))
        if (batch_num + 1) % CHECKPOINT_EVERY == 0:
            np.save(CHECKPOINT_NPY, np.array(embeddings_list, dtype=np.float32))
            df.iloc[: len(embeddings_list)][["pidp"]].to_csv(CHECKPOINT_IDX, index=False)
            pbar.set_postfix({"checkpoint": len(embeddings_list)})

# ── Save outputs ──────────────────────────────────────────────────────────────
embeddings = np.array(embeddings_list, dtype=np.float32)
assert embeddings.shape == (len(df), EMBED_DIMS), (
    f"Shape mismatch: {embeddings.shape} vs expected ({len(df)}, {EMBED_DIMS})"
)

np.save(OUTPUT_NPY, embeddings)
print(f"Saved numpy array:   {OUTPUT_NPY}  shape={embeddings.shape}")

index_df = df[["pidp"]].copy()
index_df["embedding_row"] = index_df.index
index_df.to_csv(OUTPUT_IDX_CSV, index=False)
print(f"Saved index CSV:     {OUTPUT_IDX_CSV}")

out_df = df[["pidp", "nl_profile"]].copy()
out_df["embedding"] = [row.tolist() for row in embeddings]
out_df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved parquet:       {OUTPUT_PARQUET}")

out_df.to_pickle(OUTPUT_PKL, protocol=5)
print(f"Saved pickle:        {OUTPUT_PKL}")
_csv = out_df.copy()
_csv["embedding"] = _csv["embedding"].apply(lambda e: json.dumps(e) if e is not None else "")
_csv.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV:           {OUTPUT_CSV}")

for _f in [CHECKPOINT_NPY, CHECKPOINT_IDX]:
    if _f.exists():
        _f.unlink()

print(f"\nDone. {embeddings.shape}  dtype={embeddings.dtype}  ~{embeddings.nbytes / 1e6:.1f} MB uncompressed")

Loaded .env
Reading: ../data/4_feature_eng/k_feature_eng.pkl
Loaded 27,330 profiles

Model: text-embedding-3-small  |  Profiles: 27,330  |  Est. cost: ~$0.044 USD
Starting fresh


Embedding: 100%|██████████| 27330/27330 [01:07<00:00, 403.80profile/s, checkpoint=25600]


Saved numpy array:   ../data/6_embeddings/embeddings.npy  shape=(27330, 1536)
Saved index CSV:     ../data/6_embeddings/embeddings_index.csv
Saved parquet:       ../data/6_embeddings/embeddings.parquet
Saved pickle:        ../data/6_add_vector_embedding/k_vector_embedding.pkl
Saved CSV:           ../data/6_add_vector_embedding/k_vector_embedding.csv

Done. (27330, 1536)  dtype=float32  ~167.9 MB uncompressed
